# Cross-source comparison

Load the longest-baseline `timeseries/*.tif` from each of the three example runs (`example_s1_burst`, `example_opera_cslc`, `example_nisar`) over the same LA AOI and plot them side-by-side with a uniform color scale.

The three pipelines use different wavelengths (Sentinel-1 C-band ~5.55 cm, NISAR L-band ~24.2 cm at mode 4005 freqA), so their phase-to-meters conversion factors differ by ~4x. The whole point of the comparison is to confirm that once each pipeline converts to physical units, we get consistent outputs on the same footprint.

**Caveats**:
- Dec 2025 LA is essentially noise — no tectonic signal, no atmospheric correction, one month of time. Numeric values are a noise proxy, not a geophysical result.
- The NISAR stack has only 2 SLCs (the two cycles that cover the AOI on track 34 / frame 18 in Dec 2025), so dolphin's `velocity.tif` is a degenerate 1-pair fit and comes out empty. We plot the cumulative displacement from the single interferogram instead, and derive an "effective velocity" as displacement / time-baseline for an apples-to-apples cross-source comparison.

This notebook assumes you've already run the three per-source example notebooks (or the equivalent `sweets run ...` commands).

## Load the longest-baseline timeseries TIFF from each run

In [ ]:
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

# Each per-source example notebook writes its outputs to ./example_<source>
# relative to whatever directory it was launched from. Run this notebook from
# the same CWD as those three notebooks.
RUNS_DIR = Path(".")
SOURCES = {
    "Sentinel-1 bursts":   RUNS_DIR / "example_s1_burst",
    "OPERA CSLC":          RUNS_DIR / "example_opera_cslc",
    "NISAR GSLC":          RUNS_DIR / "example_nisar",
}

PAIR_RE = re.compile(r"^(\d{8})_(\d{8})\.tif$")

def longest_pair(ts_dir: Path) -> tuple[Path, datetime, datetime]:
    """Return the `YYYYMMDD_YYYYMMDD.tif` with the widest baseline."""
    best = None
    best_span = -1
    for p in ts_dir.glob("*.tif"):
        m = PAIR_RE.match(p.name)
        if not m:
            continue
        d1 = datetime.strptime(m.group(1), "%Y%m%d")
        d2 = datetime.strptime(m.group(2), "%Y%m%d")
        span = (d2 - d1).days
        if span > best_span:
            best_span = span
            best = (p, d1, d2)
    assert best is not None, f"no pair-baseline TIFFs in {ts_dir}"
    return best

def load_raster(path: Path):
    with rasterio.open(path) as src:
        return {
            "data": src.read(1, masked=True),
            "bounds": src.bounds,
            "unit": (src.units[0] if src.units else "?") or "?",
        }

rasters = {}
for label, root in SOURCES.items():
    ts_dir = root / "dolphin" / "timeseries"
    longest_path, d1, d2 = longest_pair(ts_dir)
    r = load_raster(longest_path)
    r.update(pair_name=longest_path.name, date_start=d1, date_end=d2, baseline_days=(d2 - d1).days)
    rasters[label] = r
    print(
        f"{label:20s} {longest_path.name}  unit={r['unit']}  baseline={r['baseline_days']}d"
    )

## Cumulative displacement (meters)

Plot each longest-baseline displacement raster with a uniform symmetric color scale so the three panels are visually comparable. The S1 and OPERA panels share the same Sentinel-1 orbit (track 71 descending) so they should look structurally similar; NISAR is on a different orbit geometry (track 34 ascending) so absolute values won't line up pixel-for-pixel, but the overall spread should be consistent.

In [ ]:
def finite_values(a):
    """Return finite unmasked values from `a` as a 1D array."""
    arr = np.ma.filled(a, np.nan) if isinstance(a, np.ma.MaskedArray) else np.asarray(a)
    vals = arr.ravel()
    return vals[np.isfinite(vals)]

def symmetric_limit(arrays, pct: float = 98.0) -> float:
    stacked = np.concatenate([finite_values(a) for a in arrays])
    if stacked.size == 0:
        return 1.0
    return float(np.nanpercentile(np.abs(stacked), pct))

ts_arrays = [r["data"] for r in rasters.values()]
tlim = symmetric_limit(ts_arrays, pct=98.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
for ax, (label, r) in zip(axes, rasters.items()):
    left, bottom, right, top = r["bounds"]
    im = ax.imshow(
        r["data"],
        vmin=-tlim, vmax=tlim,
        cmap="RdBu_r",
        extent=(left, right, bottom, top),
        aspect="equal",
    )
    ax.set_title(
        f"{label}\n{r['pair_name']}"
        f" ({r['baseline_days']}-day baseline)"
    )
    ax.set_xlabel("easting (m)")
    ax.set_ylabel("northing (m)")
fig.colorbar(im, ax=axes, shrink=0.75, label="cumulative displacement (m)")
fig.suptitle(
    "sweets cross-source cumulative displacement"
    f" — LA AOI, Dec 2025, |color|={tlim:.3f} m",
    fontsize=12,
)
plt.show()

## Effective velocity (meters / year)

Divide the longest-baseline displacement by the baseline in years. This is a back-of-the-envelope annual rate — not a proper least-squares velocity fit over a full network, but it's defined for every source regardless of stack length and lets us compare magnitudes across sources on a common axis.

In [ ]:
rasters_eff = {}
for label, r in rasters.items():
    years = r["baseline_days"] / 365.25
    rasters_eff[label] = {
        "data": r["data"] / years,
        "bounds": r["bounds"],
        "baseline_days": r["baseline_days"],
    }

vlim = symmetric_limit([r["data"] for r in rasters_eff.values()], pct=98.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
for ax, (label, r) in zip(axes, rasters_eff.items()):
    left, bottom, right, top = r["bounds"]
    im = ax.imshow(
        r["data"],
        vmin=-vlim, vmax=vlim,
        cmap="RdBu_r",
        extent=(left, right, bottom, top),
        aspect="equal",
    )
    ax.set_title(f"{label}\nannualized rate ({r['baseline_days']} d -> 1 y)")
    ax.set_xlabel("easting (m)")
    ax.set_ylabel("northing (m)")
fig.colorbar(im, ax=axes, shrink=0.75, label="effective velocity (m/yr)")
fig.suptitle(
    "sweets cross-source effective velocity"
    f" — |color|={vlim:.2f} m/yr",
    fontsize=12,
)
plt.show()

## Quick stats

In [ ]:
print(f"{'source':20s} {'baseline':>10s} {'n_valid':>10s} {'disp_std':>12s} {'eff_v_std':>12s}")
print("-" * 68)
for label, r in rasters.items():
    disp_vals = finite_values(r["data"])
    years = r["baseline_days"] / 365.25
    vel_vals = disp_vals / years if disp_vals.size else disp_vals
    print(
        f"{label:20s} {r['baseline_days']:>9d}d "
        f"{disp_vals.size:>10d} "
        f"{(disp_vals.std() if disp_vals.size else float('nan')):>9.4f} m "
        f"{(vel_vals.std() if vel_vals.size else float('nan')):>9.4f} m/yr"
    )